# HW04: ML and DL

Remember that these homework work as a completion grade. **You can skip one section without losing credit.**

## Load and Pre-process Text
We do sentiment analysis on the [Movie Review Data](https://www.cs.cornell.edu/people/pabo/movie-review-data/). If you would like to know more about the data, have a look at [the paper](https://www.cs.cornell.edu/home/llee/papers/pang-lee-stars.pdf) (but no need to do so).

In [2]:
# In this tutorial, we do sentiment analysis
# download the data
#!curl -O https://ai.stanford.edu/~amaas/data/sentiment/aclImdb_v1.tar.gz
#!tar xf aclImdb_v1.tar.gz

!curl -O https://www.cs.cornell.edu/people/pabo/movie-review-data/scale_data.tar.gz
!curl -O https://www.cs.cornell.edu/people/pabo/movie-review-data/scale_whole_review.tar.gz
 
!tar xf scale_data.tar.gz 
!tar xf scale_whole_review.tar.gz

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 3935k  100 3935k    0     0  1673k      0  0:00:02  0:00:02 --:--:-- 1677k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 8645k  100 8645k    0     0  3760k      0  0:00:02  0:00:02 --:--:-- 3763k0:07  982k


First, we have to load the data for which we provide the function below. Note how we also preprocess the text using gensim's simple_preprocess() function and how we already split the data into a train and test split.

In [4]:
import os
from gensim.utils import simple_preprocess
def load_data():
    examples, labels = [], []
    authors = os.listdir("scale_whole_review")
    for author in authors:
        path = os.listdir(os.path.join("scale_whole_review", author, "txt.parag"))
        fn_ids = os.path.join("scaledata", author, "id." + author)
        fn_ratings = os.path.join("scaledata", author, "rating." + author)
        with open(fn_ids) as ids, open(fn_ratings) as ratings:
            for idx, rating in zip(ids, ratings):
                labels.append(float(rating.strip()))
                filename_text = os.path.join("scale_whole_review", author, "txt.parag", idx.strip() + ".txt")
                with open(filename_text, encoding='latin-1') as f:
                    examples.append(" ".join(simple_preprocess(f.read())))
    return examples, labels
                  
X,y  = load_data()
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.33, random_state=42)
print ("text:", X_train[0], "\nlabel:", y_train[0])

text: bloody child the director writer cinematographer nina menkes screenwriter tinka menkes editors nina and tina menkes cast tinka menkes captain sherry sibley murdered wife robert mueller murderer russ little sergeant jack hara enlisted man runtime mirage reviewed by dennis schwartz an amazingly strange film confusing and not thoroughly enjoyable but film found more interesting than thought possible at first viewing this experimental film in minimalist story telling film consisting of disturbing visualizations and almost no dialogue had concept that was greater than how the film turned out it felt at times like was watching paint dry on the wall but the reward for sitting through those excruciatingly redundant scenes was in seeing something different something that cast spell of sorcery over terrible incident as believe the film in its unique and sometimes shrill voice does justice in commenting on the violence in american society especially against women the film uses its impressio

## Vectorize the data

In [ ]:
# train a TF_IDF Vectorizer on X_train and vectorize X_train and X_test
from sklearn.feature_extraction.text import TfidfVectorizer

vec = TfidfVectorizer(min_df=0.01, # at min 1% of docs
                        max_df=.5,  
                        stop_words='english',
                        ngram_range=(1,2))

##TODO train vectorizer
vec.fit(X_train)

##TODO transform X_train to TF-IDF values
X_train_tfidf = vec.transform(X_train)
##TODO transform X_test to TF-IDF values
X_test_tfidf = vec.transform(X_test)

In [ ]:
##TODO scale both training and test data with the standard scaler
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler(with_mean=False)

scaler.fit(X_train_tfidf)

X_train_tfidf = scaler.transform(X_train_tfidf)
X_test_tfidf = scaler.transform(X_test_tfidf)

## ElasticNet

In [8]:
##TODO train an elastic net on the transformed output of the scaler
from sklearn.linear_model import ElasticNet

en = ElasticNet(alpha=0.01)

##TODO train the ElasticNet
en.fit(X_train_tfidf, y_train)

##TODO predict the testset
y_pred = en.predict(X_test_tfidf)

from sklearn.metrics import r2_score, accuracy_score, mean_squared_error, balanced_accuracy_score
##TODO print mean squared error and r2 score on the test set
print("MSE:", mean_squared_error(y_test, y_pred))
print("R2:", r2_score(y_test, y_pred))

MSE: 0.016535782264671804
R2: 0.4977289048964286


## Logistic Regression

Next, we train an OLS model doing binary prediction on these movie reviews. Two get two bins, we transform the continuous ratings into two classes, where one class contains all the negative ratings (value < 0.5), the other class all the positive ratings (value > 0.5)

In [9]:
y_train = [1 if i >= 0.5 else 0 for i in y_train]
y_test = [1 if i >= 0.5 else 0 for i in y_test]


In [10]:
##TODO train logistic regression on X_train
from sklearn.linear_model import LogisticRegression
logistic_regression = LogisticRegression()

##TODO train a logistic regression
logistic_regression.fit(X_train_tfidf, y_train)

##TODO predict the testset 
y_pred_proba = logistic_regression.predict_proba(X_test_tfidf)[:, 1]

##since we have continuous output, we need to post-process our labels into two classes. We choose a threshold of 0.5 
def map_predictions(predicted):
    predicted = [1 if i > 0.5 else 0 for i in predicted]
    return predicted

y_pred_binary = map_predictions(y_pred_proba)

##TODO print the accuracy of our classifier on the testset
print("Accuracy:", accuracy_score(y_test, y_pred_binary))

## TODO print the 10 most informative words of the regression (the 10 words having the highest coefficients)
feature_names = vec.get_feature_names_out()
top10_idx = logistic_regression.coef_[0].argsort()[-10:][::-1]
print("Top 10 most informative words:", [feature_names[i] for i in top10_idx])

Accuracy: 0.8111380145278451
Top 10 most informative words: ['great', 'surprisingly', 'best', 'effective', 'fascinating', 'success', 'punches', 'elements', 'brilliant', 'fine']


# Deep Learning

## MLP

In [11]:
#Import the AG news dataset (same as hw01)
#Download them from here 
#!wget https://raw.githubusercontent.com/mhjabreel/CharCnn_Keras/master/data/ag_news_csv/train.csv

import pandas as pd
import nltk
df = pd.read_csv('train.csv')

df.columns = ["label", "title", "lead"]
label_map = {1:"world", 2:"sport", 3:"business", 4:"sci/tech"}
def replace_label(x):
	return label_map[x]
df["label"] = df["label"].apply(replace_label) 
df["text"] = df["title"] + " " + df["lead"]
df = df.sample(n=10000) # # only use 10K datapoints
df.head()

,label,title,lead,text
55945,sport,Glazer set to seize Man Utd,AMERICAN sports magnate Malcolm Glazer is pois...,Glazer set to seize Man Utd AMERICAN sports ma...
50681,sci/tech,Ballmer sees squeeze on Longhorn deadline,Microsoft's CEO lends some insight into the 20...,Ballmer sees squeeze on Longhorn deadline Micr...
5817,sport,David Davies gets swimming bronze,Australia #39;s Grant Hackett has won the 1500...,David Davies gets swimming bronze Australia #3...
56369,world,France-China deals awaken Europeans,PARIS President Jacques Chirac of France will ...,France-China deals awaken Europeans PARIS Pres...
46256,sci/tech,Mobiles get magnifier software,Magnification software to help visually impair...,Mobiles get magnifier software Magnification s...


In [14]:
# create a new variable "business" that takes value 1 if the label is business and 0 otherwise
df['business'] = df['label'].apply(lambda x: int(x=='business'))
y = df['business'].values
df['business'].head()

55945    0
50681    0
5817     0
56369    0
46256    0
Name: business, dtype: int64

In [15]:
import spacy
nlp = spacy.load('en_core_web_sm')
from sklearn.feature_extraction.text import CountVectorizer

# pre-process text as you did in HW02
def tokenize(x):
    return [w.lemma_.lower() for w in nlp(x) if not w.is_stop and not w.is_punct and not w.is_digit]
df["tokens"] = df["text"].apply(lambda x: tokenize(x))
df["preprocessed"] = df['tokens'].apply(lambda x: ' '.join(x))
df["preprocessed_text"] = df["preprocessed"].apply(lambda x: " ".join(x))

##TODO vectorize the pre-processed text using CountVectorizer
cv = CountVectorizer(min_df=0.01, max_df=0.5)
X = cv.fit_transform(df["preprocessed"]).toarray()

Your goal here is to use features from the Vectorized text to predict whether the snippet is from a business article.

In [16]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from torchsummary import summary

import math
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

## TODO build a MLP model with at least 2 hidden layers with ReLU activation, followed by dropout and an output layer with sigmoid activation
input_dim = X.shape[1]

class MLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1 = nn.Linear(input_dim, 256)
        self.fc2 = nn.Linear(256, 64)
        self.dropout = nn.Dropout(0.3)
        self.out = nn.Linear(64, 1)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        return torch.sigmoid(self.out(x))

model = MLP(input_dim)

## TODO summarize the model using torchsummary
summary(model, (input_dim,))

# train/val/test split
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=42)
X_tr, X_val, y_tr, y_val = train_test_split(X_tr, y_tr, test_size=0.1, random_state=42)

X_tr_t   = torch.FloatTensor(X_tr)
y_tr_t   = torch.FloatTensor(y_tr)
X_val_t  = torch.FloatTensor(X_val)
y_val_t  = torch.FloatTensor(y_val)
X_te_t   = torch.FloatTensor(X_te)

train_loader = DataLoader(torch.utils.data.TensorDataset(X_tr_t, y_tr_t), batch_size=32, shuffle=True)

## TODO fit the model using early stopping to predict the business label
# (hint: early stopping means if the validation score does not increase for more than "patience" times, training should stop and load the best model so far)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
criterion = nn.BCELoss()
patience = 5
best_val_loss = float('inf')
patience_counter = 0
best_state = None

for epoch in range(100):
    model.train()
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X_batch).squeeze(), y_batch)
        loss.backward()
        optimizer.step()

    model.eval()
    with torch.no_grad():
        val_loss = criterion(model(X_val_t).squeeze(), y_val_t).item()

    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
        best_state = {k: v.clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"Early stopping at epoch {epoch + 1}")
            break

model.load_state_dict(best_state)

model.eval()
with torch.no_grad():
    test_preds = (model(X_te_t).squeeze() > 0.5).int().numpy()

print("Test Accuracy:", accuracy_score(y_te, test_preds))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                  [-1, 256]          96,768
           Dropout-2                  [-1, 256]               0
            Linear-3                   [-1, 64]          16,448
           Dropout-4                   [-1, 64]               0
            Linear-5                    [-1, 1]              65
Total params: 113,281
Trainable params: 113,281
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.43
Estimated Total Size (MB): 0.44
----------------------------------------------------------------
Early stopping at epoch 6
Test Accuracy: 0.894
